# Model Serving with FastAPI — হাতে-কলমে

এই notebook-এ আমরা দেখব: (১) blocking call কেন server আটকায়, (২) threadpool কীভাবে বাঁচায়, (৩) `rag-api` TestClient দিয়ে চালানো, (৪) throughput vs latency। কিছুই Ollama লাগে না।

In [ ]:
import sys, time, asyncio
from pathlib import Path
sys.path.insert(0, str(Path("../Class 1 Project").resolve()))

## 1. Blocking call event loop আটকায়

`/ask` ০.৫s blocking কাজ করে। একই সময়ে `/health` কত দেরিতে উত্তর পায়?

In [ ]:
from fastapi.concurrency import run_in_threadpool

def llm_call():            # Ollama-র মতো blocking
    time.sleep(0.5)

async def bad_ask():       # HIGHLIGHTS: সরাসরি blocking call — loop freeze
    llm_call()

async def good_ask():      # HIGHLIGHTS: threadpool-এ পাঠানো — loop মুক্ত
    await run_in_threadpool(llm_call)

async def measure(ask):
    t_health = None
    async def health():
        nonlocal t_health
        await asyncio.sleep(0.05)          # /health ০.০৫s পরে আসে
        t0 = time.perf_counter(); await asyncio.sleep(0); t_health = time.perf_counter()
        return t_health
    start = time.perf_counter()
    await asyncio.gather(ask(), health())
    return t_health - start

print("blocking  : /health উত্তর পেল", round(asyncio.run(measure(bad_ask)), 2), "s পরে")
print("threadpool: /health উত্তর পেল", round(asyncio.run(measure(good_ask)), 2), "s পরে")

প্রথমটায় `/health` ~০.৫s আটকে থাকে (loop freeze), দ্বিতীয়টায় ~০.০৫s।

## 2. rag-api TestClient দিয়ে (fake backend)

In [ ]:
from fastapi.testclient import TestClient
from app.main import app, get_service
from app.backend import FakeBackend
from app.service import RagService

svc = RagService(FakeBackend())
records = []
svc.observers.append(records.append)     # HIGHLIGHTS: Class 30 seam — প্রতি call-এর CallRecord এখানে আসে
app.dependency_overrides[get_service] = lambda: svc

with TestClient(app) as c:
    print(c.post("/upload", files={"file": ("hr.pdf", b"%PDF-1.4 x", "application/pdf")}).json())
    print(c.post("/ask", json={"question": "leave policy?"}).json())
    print(c.post("/ask", json={"question": ""}).json())            # validation error
    print(c.post("/upload", files={"file": ("a.txt", b"x", "text/plain")}).json())
for r in records: print(r.op, r.ok, round(r.latency_s, 4))

## 3. Throughput vs Latency sweep (নকল ০.২s LLM)

নিচের reference table: কোন ASGI server কখন।

| Server | Use case | Advantage | Limitation |
| --- | --- | --- | --- |
| Uvicorn (chosen) | dev + single-node | সহজ, দ্রুত | multi-process নিজে manage করে না |
| Gunicorn + Uvicorn | production multi-core | supervision | Unix only |
| Hypercorn | HTTP/2, HTTP/3 | trio support | ecosystem ছোট |
| Granian | Rust-based | কম overhead | কম battle-tested |

In [ ]:
import httpx, statistics

async def sweep(levels=(1, 4, 16, 64), total=64):
    svc = RagService(FakeBackend(delay=0.2))
    app.dependency_overrides[get_service] = lambda: svc
    out = []
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://t") as cl:
        for conc in levels:
            sem = asyncio.Semaphore(conc)
            async def one(i):
                async with sem:
                    t0 = time.perf_counter()
                    await cl.post("/ask", json={"question": f"q{i}"})
                    return time.perf_counter() - t0
            t0 = time.perf_counter()
            lats = sorted(await asyncio.gather(*(one(i) for i in range(total))))
            wall = time.perf_counter() - t0
            out.append((conc, total / wall, statistics.median(lats), lats[int(.95 * (total - 1))]))
    return out

res = await sweep()
print("conc  req/s   p50    p95")
for c, rps, p50, p95 in res: print(f"{c:>4} {rps:>7.1f} {p50:>6.3f} {p95:>6.3f}")

**পড়ুন:** concurrency ১ → ৪ → ১৬-তে throughput প্রায় সমানুপাতিক বাড়ে, latency প্রায় স্থির। ৬৪-তে (threadpool ৪০ ছাড়িয়ে) throughput আর সমানুপাতিক নয়, p95 বাড়ে — এটাই knee। Real Ollama-তে knee আরো আগে আসবে।

## Exercise

`FakeBackend(delay=0.2)`-কে `delay=1.0` করে levels `(1, 8, 40, 80)` চালান। knee কি সরল?